# VigiLens — Full VidLLVIP Training & Evaluation (Google Colab T4)

This notebook runs the complete VigiLens pipeline on **all 894 paired RGB + IR clips** from **VidLLVIP** while keeping the **ImageNet-pretrained ResNet-50 and YOLOv8n-seg weights frozen as-is**:

1. **Setup & GPU check** (Tesla T4, FP16 AMP)
2. **Optional Google Drive backup**
3. **Download & extract VidLLVIP** (6.18 GB, all 894 clips)
4. **Scene-disjoint 3-way split** (`train` / `val` / `test` — zero frame leakage across splits)
5. **Single-pass feature precomputation** (YOLOv8n-seg + frozen ResNet-50, saved in float16)
6. **Train Fusion + Causal LSTM** on all cached training features + plot loss curves
7. **Calibrate anomaly threshold** (`μ + 3σ` and `p95`) on held-out normal `val` clips
8. **Test & evaluate** on unseen `test` clips (`RGB+IR`, `RGB-only`, and `IR-only` ablation)
9. **End-to-end streaming test** (`InferencePipeline` on raw `.mp4` videos + temporal perturbation sanity check)
10. **Export checkpoints & evaluation results** (`best_model.pth`, `calibration.json`, `results.json`, `scores.csv`)

**Runtime:** Select `Runtime -> Change runtime type -> T4 GPU` before running.

## 1. GPU Check

In [ ]:
import json
import os
import shutil
from pathlib import Path

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
if DEVICE == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: No CUDA GPU detected. Select Runtime -> Change runtime type -> T4 GPU.")

## 2. Clone Repository & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/soumyaranjan4446/VigiLens.git"
PROJECT_DIR = "/content/VigiLens"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull --ff-only

%cd {PROJECT_DIR}
!git rev-parse --short HEAD

In [ ]:
# Colab already includes CUDA-enabled torch/torchvision.
!pip install -q -r requirements.txt

import cv2, sklearn, streamlit, ultralytics
print("opencv     :", cv2.__version__)
print("ultralytics:", ultralytics.__version__)
print("sklearn    :", sklearn.__version__)

## 3. Optional Google Drive Backup

Set `USE_DRIVE = True` to save checkpoints, evaluation results, and (optionally) the precomputed feature cache to Google Drive so you can resume if Colab disconnects.
Training and precomputing still run on Colab's fast local SSD (`/content/VigiLens/`) and sync to Drive afterward.

In [ ]:
USE_DRIVE = False  # Set True to mount Google Drive and persist outputs
DRIVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/VigiLens_runs/vidllvip_full")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive backup directory:", DRIVE_DIR)
else:
    print("Drive backup disabled. Set USE_DRIVE = True to enable it.")

## 4. Download & Prepare Entire VidLLVIP Dataset (All 894 Clips)

We split VidLLVIP **strictly by scene ID** into three disjoint sets:
- **Train (~70% of scenes):** Used to train Fusion + Causal LSTM
- **Val (~15% of scenes):** Used for early stopping (`best_model.pth`) and threshold calibration (`μ + 3σ`)
- **Test (~15% of scenes):** Held completely unseen until final evaluation (`Evaluation/manifest_vidllvip.json`)

In [ ]:
# Download VidLLVIP dataset.zip (6.18 GB) from Hugging Face and extract
!wget -c -O /content/dataset.zip "https://huggingface.co/datasets/jianfeng0369/VidLLVIP/resolve/main/dataset/dataset.zip"
!unzip -q -o /content/dataset.zip -d /content/VidLLVIP
print("VidLLVIP extracted.")

In [ ]:
# Prepare all 894 VidLLVIP clips into disjoint train / val / test splits by scene
!rm -rf Data cache checkpoints Evaluation/manifest_vidllvip.json Evaluation/results*

!python scripts/prepare_vidllvip.py \
    --input /content/VidLLVIP \
    --output Data \
    --val-ratio 0.15 \
    --test-ratio 0.15 \
    --seed 42 \
    --eval-manifest Evaluation/manifest_vidllvip.json

for split in ["train", "val", "test"]:
    rgb_clips = sorted((Path("Data") / split / "rgb").glob("*.mp4"))
    scenes = sorted({p.stem.split("_")[0] for p in rgb_clips})
    print(f"{split:5s}: {len(rgb_clips):3d} clips | scenes: {scenes}")

## 5. Precompute Features Once (YOLOv8n-seg + Frozen ImageNet ResNet-50)

Runs YOLOv8n-seg + dual frozen ResNet-50 encoders once across `train`, `val`, and `test`.
`--save-half` stores float16 features (~900 MB total for all 894 clips instead of ~1.8 GB). Expect ~15–25 minutes on a T4 GPU.
Omitting `--overwrite` makes this cell safe to resume if interrupted.

In [ ]:
!python -m training.precompute \
    --data-dir Data \
    --output cache \
    --splits train,val,test \
    --device {DEVICE} \
    --batch-size 32 \
    --save-half

for split in ["train", "val", "test"]:
    pt_files = list((Path("cache") / split).glob("*.pt"))
    print(f"cache/{split:5s}: {len(pt_files)} files")

## 6. Train Fusion + Causal LSTM (30 Epochs)

ResNet-50 stays frozen; only `ModalityAwareFusion` and `TemporalModel` (3.09M parameters) train.
`--max-cached-videos 1024` keeps the entire training cache in system RAM (~700 MB) so every epoch runs in a few seconds on T4 GPU.
`--calibrate` automatically runs threshold calibration on `cache/val` using `best_model.pth` once training completes.

In [ ]:
!python -m training.train \
    --cache-dir cache \
    --epochs 30 \
    --batch-size 64 \
    --val-batch-size 64 \
    --seq-len 16 \
    --lr 1e-4 \
    --weight-decay 1e-5 \
    --grad-clip 1.0 \
    --max-cached-videos 1024 \
    --device {DEVICE} \
    --checkpoint-dir checkpoints \
    --calibrate

In [ ]:
# Plot training & validation loss curves
import matplotlib.pyplot as plt

history_path = Path("checkpoints/training_history.json")
if not history_path.exists():
    raise RuntimeError("checkpoints/training_history.json not found. Run training first.")

history = json.loads(history_path.read_text(encoding="utf-8"))
epochs = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
lrs = [h["lr"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, train_loss, label="Train Loss", marker="o", markersize=3)
axes[0].plot(epochs, val_loss, label="Val Loss", marker="s", markersize=3)
axes[0].set_title("VigiLens Training Convergence (Full VidLLVIP)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Smooth L1 Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.4)

axes[1].semilogy(epochs, lrs, label="Learning Rate", marker="^", markersize=3, color="green")
axes[1].set_title("Learning Rate Schedule")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("checkpoints/training_curves.png", dpi=120)
plt.show()

best_epoch = min(history, key=lambda h: h["val_loss"])
print(f"Best checkpoint: epoch {best_epoch['epoch']} | val_loss={best_epoch['val_loss']:.6f}")

## 7. Calibrate Threshold on Held-Out Normal Validation Set (`cache/val`)

Computes `threshold = mean_error + 3.0 * std_error` (and stores `p95_error`) from `checkpoints/best_model.pth` on `cache/val`.

In [ ]:
if not Path("checkpoints/best_model.pth").exists():
    raise RuntimeError("Cannot find checkpoints/best_model.pth! Run training first.")

!python -m training.calibration \
    --cache-dir cache/val \
    --checkpoint checkpoints/best_model.pth \
    --output checkpoints/calibration.json \
    --device {DEVICE} \
    --k-std 3.0 \
    --percentile 95

cal = json.loads(Path("checkpoints/calibration.json").read_text(encoding="utf-8"))
print(f"\nCalibrated threshold (mu + 3*sigma): {cal['threshold']:.6f}")
print(f"Mean error: {cal['mean_error']:.6f} | Std error: {cal['std_error']:.6f} | Samples: {cal['num_samples']}")

## 8. Test & Evaluate on Unseen Test Split (`cache/test`)

Evaluates `checkpoints/best_model.pth` + `checkpoints/calibration.json` on the completely unseen VidLLVIP `test` scenes across three sensor configurations:
1. **RGB + IR** (full multimodal)
2. **RGB-only** (simulates IR camera outage)
3. **IR-only** (simulates visible camera outage / total darkness)

> Because VidLLVIP contains normal pedestrian surveillance scenes only, `AUC` and `AP` are `N/A` (single class), while **False Positive Rate (FPR)**, **Specificity**, and **score stability** directly measure real-world normal baseline accuracy.

In [ ]:
!python -m training.evaluate \
    --manifest Evaluation/manifest_vidllvip.json \
    --cache-dir cache/test \
    --checkpoint checkpoints/best_model.pth \
    --calibration checkpoints/calibration.json \
    --output-dir Evaluation/results \
    --modality auto \
    --device {DEVICE} \
    --smoothing-window 5

!python -m training.evaluate \
    --manifest Evaluation/manifest_vidllvip.json \
    --cache-dir cache/test \
    --checkpoint checkpoints/best_model.pth \
    --calibration checkpoints/calibration.json \
    --output-dir Evaluation/results_rgb_only \
    --modality rgb \
    --device {DEVICE} \
    --smoothing-window 5

!python -m training.evaluate \
    --manifest Evaluation/manifest_vidllvip.json \
    --cache-dir cache/test \
    --checkpoint checkpoints/best_model.pth \
    --calibration checkpoints/calibration.json \
    --output-dir Evaluation/results_ir_only \
    --modality ir \
    --device {DEVICE} \
    --smoothing-window 5

In [ ]:
# Summarize metrics across modalities and plot score distributions
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for name, res_dir in [
    ("RGB + IR", "Evaluation/results"),
    ("RGB-only", "Evaluation/results_rgb_only"),
    ("IR-only",  "Evaluation/results_ir_only"),
]:
    res_file = Path(res_dir) / "results.json"
    scores_file = Path(res_dir) / "scores.csv"
    if res_file.exists():
        r = json.loads(res_file.read_text(encoding="utf-8"))
        m = r["metrics"]
        df_s = pd.read_csv(scores_file)
        rows.append({
            "Modality": name,
            "Videos": r["num_videos"],
            "Frames": r["num_frames"],
            "TN": m["tn"],
            "FP": m["fp"],
            "FPR (%)": round(m["fpr"] * 100, 3),
            "Specificity (%)": round((1.0 - m["fpr"]) * 100, 3),
            "Mean Score": round(float(df_s["smoothed_score"].mean()), 6),
            "Max Score": round(float(df_s["smoothed_score"].max()), 6),
            "Threshold": round(float(r["threshold"]), 6),
        })

summary_df = pd.DataFrame(rows)
display(summary_df)

scores_mm = pd.read_csv("Evaluation/results/scores.csv")
threshold = rows[0]["Threshold"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scores_mm["smoothed_score"], bins=60, color="steelblue", alpha=0.8)
axes[0].axvline(threshold, color="red", linestyle="--", linewidth=2, label=f"Threshold = {threshold:.6f}")
axes[0].set_title("Unseen Test Frame Scores (RGB + IR)")
axes[0].set_xlabel("Smoothed Anomaly Score")
axes[0].set_ylabel("Frame Count")
axes[0].legend()
axes[0].grid(True, alpha=0.4)

per_clip = scores_mm.groupby("sequence_id")["smoothed_score"].agg(["mean", "max"])
axes[1].plot(range(len(per_clip)), per_clip["mean"].values, label="Clip Mean", marker="o", markersize=3)
axes[1].plot(range(len(per_clip)), per_clip["max"].values, label="Clip Max", alpha=0.7)
axes[1].axhline(threshold, color="red", linestyle="--", linewidth=2, label=f"Threshold = {threshold:.6f}")
axes[1].set_title("Per-Clip Score Summary Across Unseen Test Set")
axes[1].set_xlabel("Test Clip Index")
axes[1].set_ylabel("Smoothed Score")
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("Evaluation/results/test_evaluation_plots.png", dpi=120)
plt.show()

## 9. End-to-End Streaming Test (`InferencePipeline` on Raw `.mp4` Videos)

Verifies the full real-time pipeline (`YOLOSegmentation` -> `ResNet-50` -> `ModalityAwareFusion` -> `CausalLSTM` -> `calibration.json`) directly on raw `.mp4` video pairs—matching how `app.py` executes at deployment:
1. **Unseen normal test clip** from `Data/test`
2. **Temporal-perturbation sensitivity sanity check** (creates a test video whose second half jumps/skips frames abruptly to confirm the trained detector responds to abnormal motion dynamics)

> Note: The temporal-perturbation clip is a sanity check for detector sensitivity, not a real-world anomaly benchmark. To measure real `ROC-AUC` / `AP` / `F1`, place labeled anomaly clips in `Evaluation/normal` and `Evaluation/anomaly` and run `scripts/make_eval_manifest.py`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from models.anomaly_model import VigiLensModel
from models.segmentation import YOLOSegmentation
from pipeline.frame_source import create_frame_source
from pipeline.inference import InferencePipeline
from training.calibration import AnomalyCalibrator

test_rgb_path = sorted((Path("Data/test/rgb")).glob("*.mp4"))[0]
seq_id = test_rgb_path.stem.removesuffix("_rgb")
test_ir_path = Path("Data/test/ir") / f"{seq_id}_ir.mp4"
print(f"Testing end-to-end streaming inference on unseen clip: {seq_id}")

model = VigiLensModel(device=DEVICE, load_extractors=True)
model.load_checkpoint("checkpoints/best_model.pth")
model.eval()

calibration_info = AnomalyCalibrator.load_calibration("checkpoints/calibration.json")
segmentation = YOLOSegmentation(device=DEVICE)

pipeline = InferencePipeline(
    model=model,
    segmentation=segmentation,
    calibration_info=calibration_info,
    smoothing_window=5,
)

with create_frame_source(str(test_rgb_path), str(test_ir_path)) as src:
    normal_run = pipeline.process_video(src)

print(f"Normal clip   -> frames={normal_run['total_frames']}, anomalies={normal_run['anomaly_count']}, FPS={normal_run['fps']:.1f}")

# Build a temporal-perturbation sanity clip (normal first half, 6x fast-forward + vertical flip second half)
perturbed_rgb = Path("/content/sanity_perturbed_rgb.mp4")
perturbed_ir = Path("/content/sanity_perturbed_ir.mp4")

def _make_perturbed(src_path, dst_path, is_ir=False):
    cap = cv2.VideoCapture(str(src_path))
    frames = []
    while True:
        ret, frm = cap.read()
        if not ret:
            break
        frames.append(frm)
    cap.release()
    h, w = frames[0].shape[:2]
    out = cv2.VideoWriter(str(dst_path), cv2.VideoWriter_fourcc(*"mp4v"), 25.0, (w, h))
    mid = len(frames) // 2
    for frm in frames[:mid]:
        out.write(frm)
    for idx in range(mid, len(frames)):
        # Rapid temporal jump + inversion in second half to simulate abnormal change
        jumped = frames[(mid + (idx - mid) * 6) % len(frames)]
        out.write(cv2.flip(jumped, 0))
    out.release()

_make_perturbed(test_rgb_path, perturbed_rgb)
_make_perturbed(test_ir_path, perturbed_ir, is_ir=True)

with create_frame_source(str(perturbed_rgb), str(perturbed_ir)) as src:
    sanity_run = pipeline.process_video(src)

print(f"Perturbed clip-> frames={sanity_run['total_frames']}, anomalies={sanity_run['anomaly_count']}, FPS={sanity_run['fps']:.1f}")

normal_scores = [r["smoothed_score"] for r in normal_run["results"]]
sanity_scores = [r["smoothed_score"] for r in sanity_run["results"]]

plt.figure(figsize=(10, 4))
plt.plot(normal_scores, label=f"Unseen Normal ({seq_id})", color="steelblue")
plt.plot(sanity_scores, label="Perturbed Sanity Clip (jump @ mid)", color="tomato")
plt.axhline(pipeline.threshold, color="black", linestyle="--", label=f"Threshold = {pipeline.threshold:.6f}")
plt.axvline(len(normal_scores) // 2, color="gray", linestyle=":", label="Perturbation Start")
plt.title("Streaming InferencePipeline Score Trajectory")
plt.xlabel("Frame Index")
plt.ylabel("Smoothed Anomaly Score")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("Evaluation/results/streaming_sanity_check.png", dpi=120)
plt.show()

## 10. Package & Export Checkpoints and Results

Download `/content/VigiLens_VidLLVIP_full.zip` and extract `checkpoints/best_model.pth` and `checkpoints/calibration.json` into your local `VigiLens/checkpoints/` directory to run `streamlit run app.py` locally.

In [ ]:
!zip -r /content/VigiLens_VidLLVIP_full.zip \
    checkpoints/best_model.pth \
    checkpoints/last_model.pth \
    checkpoints/calibration.json \
    checkpoints/training_history.json \
    checkpoints/training_curves.png \
    Evaluation/results \
    Evaluation/results_rgb_only \
    Evaluation/results_ir_only

print("\nReady to download: /content/VigiLens_VidLLVIP_full.zip")

if USE_DRIVE and DRIVE_DIR is not None:
    shutil.copy2("/content/VigiLens_VidLLVIP_full.zip", DRIVE_DIR / "VigiLens_VidLLVIP_full.zip")
    for folder in ["checkpoints", "Evaluation/results", "Evaluation/results_rgb_only", "Evaluation/results_ir_only"]:
        src = Path(folder)
        dst = DRIVE_DIR / folder
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
    print("Also exported checkpoints & results to Google Drive:", DRIVE_DIR)

## Optional: Evaluate on Custom Labeled Anomaly Videos (ROC-AUC / AP / F1)

If you have labeled anomaly videos (e.g., from ShanghaiTech, Avenue, or your own recordings), place normal clips under `Evaluation/normal/` and anomalous clips under `Evaluation/anomaly/` (with `segments.json` or frame-level `.txt` labels), then run:

```bash
!python scripts/make_eval_manifest.py --root Evaluation --segments Evaluation/segments.json
!python -m training.precompute --manifest Evaluation/manifest.json --output cache/eval_custom --device cuda --batch-size 32 --save-half
!python -m training.evaluate --manifest Evaluation/manifest.json --cache-dir cache/eval_custom --checkpoint checkpoints/best_model.pth --calibration checkpoints/calibration.json --output-dir Evaluation/results_custom --device cuda --smoothing-window 5
```